# Synergy model interpretation for individual drug combinations

## Data loading

Configure root with local/colab.

In [ ]:
import sys, subprocess
import pandas as pd
%matplotlib inline
from pathlib import Path

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/phenotype-prediction.git"
repo_dir = Path("phenotype-prediction")
if COLAB:
    root = Path("/content/phenotype-prediction")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

Configure data path for colab/local.

In [ ]:
if COLAB:
  from google.colab import drive
  drive.mount("/content/drive")
  data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
  fcnts_path = str(data_dir / "fcnts_timezero")
  cfu_path = str(data_dir /  "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

else:
  data_dir = Path("C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab")
  fcnts_path = str(data_dir / "fcnts_timezero")
  cfu_path = str(data_dir / "cfus")
  annot_path = str(data_dir / "Annotation_TIGR4.tsv")

Load the time-matched transcriptional scores, synergy scores, and metadata.

In [ ]:
from src.dge_data import (
    simple_interaction_score,
    eob_score,
    get_all_synergy_data
)

# Bliss score and simple interaction score
data_df = get_all_synergy_data(
    l2fc_dir = l2fc_dir,
    cfu_dir = cfu_dir,
    interaction_score_method = simple_interaction_score,
    synergy_score_method = eob_score,
    time_matched = True
)

# Drop genes with NA values
data_df = data_df.dropna(axis = 1)

## CEF+CIP feature interpretation

Feature importance strategy:
- Train 5-fold CV 
- Find features with consistently high coefficients (mean / std) -> candidate mechanistic contributors to prediction
- Find expression levels of these genes in the original data (heatmap)
- Once candidates identified, use statistical testing in low and high synergy groups?

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import GridSearchCV
from src.split import random_combination_splits

def cv_feature_importances(
    df: pd.DataFrame,
    splits: list[tuple[list, list]]
):
    """
    Running nested CV and gather feature importances across folds
    """
    # Initialize feature importance dataframe
    feature_df = []

    # Nested cross-validation to train models
    for train_idx, _ in splits:
        train_df = df.iloc[train_idx]
        X_train = train_df.iloc[:, train_df.columns.str.contains("SP")]
        y_train = train_df["synergy_score"]

        param_grid = {
                "model__n_components": list(range(3, 20))
        }

        pipeline = Pipeline([
                ("scaler", StandardScaler()),
                ("model", PLSRegression())
        ])

        search = GridSearchCV(
                estimator = pipeline,
                cv = 5,
                param_grid = param_grid,
                scoring = "neg_mean_squared_error",
        )

        search.fit(X_train, y_train)
        best_pipeline = search.best_estimator_

        # Add store parameters in feature importances
        coefs = best_pipeline.named_steps["model"].coef_.ravel()
        feature_df.append(coefs)
    
    # Reshape to dataframe
    feature_df = pd.DataFrame(feature_df).T
    feature_df.index = df.columns[df.columns.str.contains("SP")]

    # Calculate importances
    feature_df["importance"] = feature_df.mean(axis = 1) / feature_df.std(axis = 1)
    feature_df = feature_df["importance"].to_frame()

    return feature_df

Train and extract features.

In [ ]:
from sklearn.model_selection import KFold

# Random splits
cv = KFold(
    n_splits = 5,
    shuffle = True,
    random_state = 111
)

# Isolate cefcip data
cefcip_df = data_df[data_df["drug_id"] == "CEF+CIP"]

# Get splits
cefcip_splits = cv.split(cefcip_df)

# Get features
cefcip_features = cv_feature_importances(
    df = cefcip_df,
    splits = cefcip_splits
)

cefcip_features

In [ ]:
# Load annotations
annot_path = 
annotations = pd.read_table(annot_path, sep = "\t")
annotations.set_index("TIGR4.old", inplace = True, drop = True)

In [ ]:
# Load annotations
annotations = pd.read_table(annot_path, sep = "\t")
annotations.set_index("TIGR4.old", inplace = True, drop = True)